Univerzitet u Sarajevu
<br> Elektrotehnički fakultet
#### **Predmet: Multimedijalni Sistemi**

# Laboratorijska vježba 02 :  Shannon-Fano algoritam kodiranja

---

Za izradu laboratorijske vježbe treba koristiti odgovarajuću Jupyter Notebook datoteku. Urađenu vježbu je potrebno konvertirati u PDF format, a zatim je PDF datoteku potrebno predati do postavljenog roka koristeći platformu Zamger.

Ime i prezime studenta, broj indeksa: Ismar Muslić, 2366/19304

Datum izrade izvještaja: 22.3.2026.

---

## Zadatak 1.

Potrebno je realizirati Shannon-Fano algoritam kodiranja putem funkcije <code>shannon_fano</code> koja kao parametar prima neku tekstualnu poruku, a kao rezultat vraća listu simbola i listu svih kodova dodijeljenih pojedinačnim simbolima. Potrebno je implementirati i funkcije <code>kodiraj</code> i <code>entropija</code>. Funkcija <code>kodiraj</code> kao parametre prima poruku, listu simbola i listu kodova, a kao rezultat vraća kodiranu poruku. Funkcija <code>entropija</code> kao parametar prima poruku, a kao rezultat vraća entropiju za tu poruku.

Shannon-Fano algoritam predstavlja osnovni algoritam kodiranja na bazi vjerovatnoće pojavljivanja simbola u poruci. Sastoji se iz sljedećih koraka:

1. Za zadanu listu simbola, izračunati vjerovatnoće pojavljivanja u poruci.

2. Sortirati simbole prema frekvenciji od najveće do najmanje frekvencije.

3. Podijeliti listu na dva dijela, po mogućnosti sa jednakim sumama vjerovatnoća ponavljanja.

4. Lijeva ivica dobiva vrijednost 0, a desna 1 (svi kodovi koji počinju sa simbolima na lijevoj strani počinju sa 0, a na desnoj sa 1).

5. Izvršavati korake 3 i 4 dok se potpuno ne izgradi binarno stablo (svi simboli postanu čvorovi stabla).

Detaljne informacije o samom algoritmu moguće je pronaći u Poglavlju 2 materijala za rad na predmetu, na str. 21 - 25.

Dobivena rješenja za kodne riječi nisu jedinstvena, tj. dobivena rješenja za kodne riječi ovise o specifičnim implementacijskim detaljima. Bitno je da implementacija bude realizirana u skladu sa gore opisanim koracima, tj. u skladu sa algoritmom opisanim na predavanjima.

**Rješenje:**

In [2]:
import math
from collections import Counter

In [3]:
def sortiranje_simbola(simboli, frekvencija):
    # sortiramo po frekvenciji u opadajucem poretku
    duzina = len(simboli)

    for i in range(duzina-1):
        for j in range(0, duzina-i-1):
            if frekvencija[j] < frekvencija[j+1]:
                frekvencija[j], frekvencija[j+1] = frekvencija[j+1], frekvencija[j]
                simboli[j], simboli[j+1] = simboli[j+1], simboli[j]

    return simboli, frekvencija # sortirana lista sa frekvencijama

In [4]:
def kodovi_simbola(frekvencija, kodovi, pocetak, kraj):
    # ukupna frekvencija na posmatranom dijelu liste
    suma_frekvencija = 0

    for i in range(pocetak, kraj):
        suma_frekvencija += frekvencija[i]

    # Pocetna min razlika - smanjivat ce se kroz petlju
    min_razlika = 1

    # Indeks mjesta podjele
    podjela = pocetak

    # Akumulirana razlika
    razlika = 0

    # Podjela liste po vjerovatnocama pojavljivanja
    for i in range(pocetak, kraj):
        temp = frekvencija[i] / suma_frekvencija # relativna frekvencija pojavljivanja simbola
        razlika += temp                          # akumulirana frekvencija

        if min_razlika > abs(razlika - 0.5):     # azuriranje razlike
            min_razlika = abs(razlika - 0.5)
            podjela = i + 1                      # azuriranje mjesta podjele

    # Dodjela kodova simbolima
    for i in range(pocetak, podjela):
        kodovi[i] += '0'

    for i in range(podjela, kraj):
        kodovi[i] += '1'

    # Rekurzivni poziv funkcije za lijevi podinterval
    if pocetak + 2 <= podjela:
        kodovi_simbola(frekvencija, kodovi, pocetak, podjela)

    # Rekurzivni poziv funkcije za desni podinterval
    if podjela + 2 <= kraj:
        kodovi_simbola(frekvencija, kodovi, podjela, kraj)

In [5]:
def shannon_fano(poruka):

    simboli = [] # sadrzi simbole iz poruke

    for i in range(len(poruka)):
        if poruka[i] not in simboli:
            simboli.append(poruka[i]) # ako trenutni simbol jos nije u listi, dodajemo ga

    # Brojimo pojavljivanje pojedinacnih simbola
    frekvencija = [poruka.count(s) for s in simboli]

    # Sortiramo simbole i frekvencije po njihovoj ucestalosti pojavljivanja - opadajuci poredak
    simboli, frekvencija = sortiranje_simbola(simboli, frekvencija)

    kodovi = [""] * len(simboli) # kodovi za svaki simbol

    # Generisanje kodova koristeci Shannon-Fano
    kodovi_simbola(frekvencija, kodovi, 0, len(simboli))

    return simboli, kodovi # lista simbola i odgovarajuci kod

In [6]:
def kodiraj(poruka, simboli, kodovi):
    kodirana = ""

    for i in range(len(poruka)):
        j = simboli.index(poruka[i]) # trenutni znak u listi simbola
        kodirana += kodovi[j]        # odgovarajuci kod

    return kodirana

In [7]:
def entropija(poruka):

    simboli = []     # jedinstveni simboli iz poruke
    for i in range(len(poruka)):
        if poruka[i] not in simboli:
            simboli.append(poruka[i])  # dodavanje simbola u listu pri prvom pojavljivanju

    frekvencija = []  # frekvencije pojavljivanja simbola
    for simbol in simboli:
        brojac = poruka.count(simbol)
        frekvencija.append(brojac)

    p = []     # vjerovatnoce pojavljivanja simbola
    for broj in frekvencija:
        p.append(broj / len(poruka))

    vr_entr = 0  # entropija

    # formula Shannonove entropije
    for vjerojatnost in p:
        vr_entr += (vjerojatnost * math.log(1/vjerojatnost, 2))


    return vr_entr

In [8]:
# funkcija za racunanje efikasnosti
# mi = H/l
# H - entropija
# l - prosjecna duzina kodne rijeci

def efikasnost_kodiranja(entropija, poruka, kodirana_poruka):

    prosjecna_duzina = round(len(kodirana_poruka)/len(poruka),3)

    return entropija/prosjecna_duzina

Nakon implementacije funkcije, potrebno je biti moguće izvršiti programski kod ispod tako da daje prikazani ispis. Osim toga, potrebno je pored poruke "CAADADBAADCAADBAADAD" dodati još četiri primjera poruka kako bi se implementacija testirala. Dodatna četiri primjera poruka trebaju obavezno biti iz sljedećeg skupa {"A", "DDDDDDDDDDDDDDDDDD", "ABBBBCDECCCCAAAAAAAAAAAAAA", "AAAAAAAAAABBBBBBCCCD"}.  

In [9]:
import math
poruka = "CAADADBAADCAADBAADAD"
print("Izvorna poruka je:")
print(poruka)

(simboli, kodovi) = shannon_fano(poruka)

print("\nPoruka se sastoji od sljedecih simbola:")
print(", ".join(map(str, simboli)))

print("\nSkup dobivenih kodnih riječi je:")
print(", ".join(map(str, kodovi)))

kodirana = kodiraj(poruka, simboli, kodovi)

for i in range(len(simboli)):
  print("\nKod simbola " + simboli[i] + " je " + kodovi [i])

print("\nKodirana poruka je:")
print(kodirana)
print("\nEntropija poruke je:")
print(round(entropija(poruka),3))
print("\nProsječna dužina kodne riječi u poruci je:")
print(round(len(kodirana)/len(poruka),3))

Izvorna poruka je:
CAADADBAADCAADBAADAD

Poruka se sastoji od sljedecih simbola:
A, D, C, B

Skup dobivenih kodnih riječi je:
0, 10, 110, 111

Kod simbola A je 0

Kod simbola D je 10

Kod simbola C je 110

Kod simbola B je 111

Kodirana poruka je:
1100010010111001011000101110010010

Entropija poruke je:
1.685

Prosječna dužina kodne riječi u poruci je:
1.7


In [10]:
# helper za ispis
def test(poruka):
    print("Izvorna poruka je:")
    print(poruka)

    (simboli, kodovi) = shannon_fano(poruka) # dodati frekvenciju nakon kodovi

    print("\nPoruka se sastoji od sljedecih simbola:")
    print(", ".join(map(str, simboli)))

    # vratiti frekvenciju iz shannon_fano
    # print("\nFrekvencije pojavljivanja simbola su:")
    # print(", ".join(map(str, frekvencija)))

    print("\nSkup dobivenih kodnih riječi je:")
    print(", ".join(map(str, kodovi)))

    kodirana = kodiraj(poruka, simboli, kodovi)

    for i in range(len(simboli)):
        print("\nKod simbola " + simboli[i] + " je " + kodovi [i])

    print("\nKodirana poruka je:")
    print(kodirana)

    print("\nEntropija poruke je:")
    entropija_poruke = round(entropija(poruka),3)
    print(entropija_poruke)

    print("\nProsječna dužina kodne riječi u poruci je:")
    print(round(len(kodirana)/len(poruka),3))

    print("\nEfikasnost kodiranja je:")
    print(round(efikasnost_kodiranja(entropija_poruke, poruka, kodirana),3))

In [11]:
test("A")

Izvorna poruka je:
A

Poruka se sastoji od sljedecih simbola:
A

Skup dobivenih kodnih riječi je:
0

Kod simbola A je 0

Kodirana poruka je:
0

Entropija poruke je:
0.0

Prosječna dužina kodne riječi u poruci je:
1.0

Efikasnost kodiranja je:
0.0


In [12]:
test("DDDDDDDDDDDDDDDDDD")

Izvorna poruka je:
DDDDDDDDDDDDDDDDDD

Poruka se sastoji od sljedecih simbola:
D

Skup dobivenih kodnih riječi je:
0

Kod simbola D je 0

Kodirana poruka je:
000000000000000000

Entropija poruke je:
0.0

Prosječna dužina kodne riječi u poruci je:
1.0

Efikasnost kodiranja je:
0.0


In [13]:
test("ABBBBCDECCCCAAAAAAAAAAAAAA")

Izvorna poruka je:
ABBBBCDECCCCAAAAAAAAAAAAAA

Poruka se sastoji od sljedecih simbola:
A, C, B, D, E

Skup dobivenih kodnih riječi je:
0, 10, 110, 1110, 1111

Kod simbola A je 0

Kod simbola C je 10

Kod simbola B je 110

Kod simbola D je 1110

Kod simbola E je 1111

Kodirana poruka je:
011011011011010111011111010101000000000000000

Entropija poruke je:
1.692

Prosječna dužina kodne riječi u poruci je:
1.731

Efikasnost kodiranja je:
0.977


In [14]:
test("AAAAAAAAAABBBBBBCCCD")

Izvorna poruka je:
AAAAAAAAAABBBBBBCCCD

Poruka se sastoji od sljedecih simbola:
A, B, C, D

Skup dobivenih kodnih riječi je:
0, 10, 110, 111

Kod simbola A je 0

Kod simbola B je 10

Kod simbola C je 110

Kod simbola D je 111

Kodirana poruka je:
0000000000101010101010110110110111

Entropija poruke je:
1.648

Prosječna dužina kodne riječi u poruci je:
1.7

Efikasnost kodiranja je:
0.969


In [15]:
test("ISMAR")

Izvorna poruka je:
ISMAR

Poruka se sastoji od sljedecih simbola:
I, S, M, A, R

Skup dobivenih kodnih riječi je:
00, 01, 100, 101, 11

Kod simbola I je 00

Kod simbola S je 01

Kod simbola M je 100

Kod simbola A je 101

Kod simbola R je 11

Kodirana poruka je:
000110010111

Entropija poruke je:
2.322

Prosječna dužina kodne riječi u poruci je:
2.4

Efikasnost kodiranja je:
0.968


In [16]:
test("LABORATORIJSKEVJEZBE")

Izvorna poruka je:
LABORATORIJSKEVJEZBE

Poruka se sastoji od sljedecih simbola:
E, A, B, O, R, J, L, T, I, S, K, V, Z

Skup dobivenih kodnih riječi je:
000, 001, 0100, 0101, 011, 100, 1010, 1011, 1100, 1101, 11100, 11101, 1111

Kod simbola E je 000

Kod simbola A je 001

Kod simbola B je 0100

Kod simbola O je 0101

Kod simbola R je 011

Kod simbola J je 100

Kod simbola L je 1010

Kod simbola T je 1011

Kod simbola I je 1100

Kod simbola S je 1101

Kod simbola K je 11100

Kod simbola V je 11101

Kod simbola Z je 1111

Kodirana poruka je:
1010001010001010110011011010101111001001101111000001110110000011110100000

Entropija poruke je:
3.584

Prosječna dužina kodne riječi u poruci je:
3.65

Efikasnost kodiranja je:
0.982


In [17]:
test("MULTIMEDIJALNISISTEMI")

Izvorna poruka je:
MULTIMEDIJALNISISTEMI

Poruka se sastoji od sljedecih simbola:
I, M, L, T, E, S, U, D, J, A, N

Skup dobivenih kodnih riječi je:
00, 010, 011, 1000, 1001, 101, 1100, 1101, 11100, 11101, 1111

Kod simbola I je 00

Kod simbola M je 010

Kod simbola L je 011

Kod simbola T je 1000

Kod simbola E je 1001

Kod simbola S je 101

Kod simbola U je 1100

Kod simbola D je 1101

Kod simbola J je 11100

Kod simbola A je 11101

Kod simbola N je 1111

Kodirana poruka je:
010110001110000001010011101001110011101011111100101001011000100101000

Entropija poruke je:
3.232

Prosječna dužina kodne riječi u poruci je:
3.286

Efikasnost kodiranja je:
0.984


In [18]:
# primjer kodiranja na najefikasniji nacin
test("ABCDEFGHIJ")
# najefikasnije je kad je najveca entropija

Izvorna poruka je:
ABCDEFGHIJ

Poruka se sastoji od sljedecih simbola:
A, B, C, D, E, F, G, H, I, J

Skup dobivenih kodnih riječi je:
000, 001, 0100, 0101, 011, 100, 101, 1100, 1101, 111

Kod simbola A je 000

Kod simbola B je 001

Kod simbola C je 0100

Kod simbola D je 0101

Kod simbola E je 011

Kod simbola F je 100

Kod simbola G je 101

Kod simbola H je 1100

Kod simbola I je 1101

Kod simbola J je 111

Kodirana poruka je:
0000010100010101110010111001101111

Entropija poruke je:
3.322

Prosječna dužina kodne riječi u poruci je:
3.4

Efikasnost kodiranja je:
0.977


In [19]:
# primjer koji je najgori za Shannon-Fano
test("AAAAAAAAAAAAAAAAAA")
# najlosije je kada nema entropije

Izvorna poruka je:
AAAAAAAAAAAAAAAAAA

Poruka se sastoji od sljedecih simbola:
A

Skup dobivenih kodnih riječi je:
0

Kod simbola A je 0

Kodirana poruka je:
000000000000000000

Entropija poruke je:
0.0

Prosječna dužina kodne riječi u poruci je:
1.0

Efikasnost kodiranja je:
0.0
